In [0]:
-- Gold fact table: orders
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.gold.fact_orders AS
SELECT 
    o.order_id,
    CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    c.customer_sk,
    o.customer_id,
    o.store_id,
    o.order_status,
    o.payment_method,
    o.channel,
    COUNT(oi.line_number) AS line_count,
    SUM(oi.quantity) AS units,
    SUM(oi.gross_amount) AS gross_amount,
    SUM(oi.discount_amount) AS discount_amount,
    SUM(oi.net_amount) AS net_item_amount,
    SUM(o.delivery_fee) AS delivery_fee,
    SUM(oi.net_amount) + SUM(o.delivery_fee) AS order_total,
    o.promised_mins,
    o.actual_mins,
    CASE WHEN o.order_status = 'DELIVERED' AND o.actual_mins IS NOT NULL AND o.promised_mins IS NOT NULL THEN (o.actual_mins <= o.promised_mins) ELSE NULL END AS is_on_time
FROM fk_ce_2026_07_1372.silver.orders o
LEFT JOIN fk_ce_2026_07_1372.silver.order_items oi
    ON o.order_id = oi.order_id
LEFT JOIN fk_ce_2026_07_1372.gold.dim_customer c
    ON o.customer_id = c.customer_id
        AND (c.effective_from IS NULL OR c.effective_from <= o.order_date)
        AND (c.effective_to IS NULL OR o.order_date < c.effective_to)
GROUP BY o.order_id, o.order_date, c.customer_sk, o.customer_id, o.store_id, o.order_status, o.payment_method, o.channel, o.promised_mins, o.actual_mins;


In [0]:
-- Gold fact table: order_line
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.gold.fact_order_line AS
SELECT
    oi.order_id,
    oi.line_number,
    CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    c.customer_sk,
    o.store_id,
    oi.product_id,
    o.order_status,
    oi.quantity,
    oi.unit_price,
    oi.gross_amount,
    oi.discount_amount,
    oi.net_amount
FROM fk_ce_2026_07_1372.silver.order_items oi
LEFT JOIN fk_ce_2026_07_1372.silver.orders o
    ON oi.order_id = o.order_id
LEFT JOIN fk_ce_2026_07_1372.gold.dim_customer c
    ON o.customer_id = c.customer_id
        AND (c.effective_from IS NULL OR c.effective_from <= o.order_date)
        AND (c.effective_to IS NULL OR o.order_date < c.effective_to);
